# Entrega — Agente Flat Monte Carlo
**Juan Gomez | Group A | Fundamentos de IA — Reto Connect-4**

Este notebook contiene el analisis empirico completo del agente `FlatMonteCarlo`.
Se evalua el rendimiento en funcion de la variable de configuracion **N** (presupuesto de rollouts),
contra el jugador aleatorio y contra si mismo, para ambos colores.

## Indice
1. Descripcion del agente
2. Experimento 1 — Win rate vs jugador aleatorio en funcion de N
3. Experimento 2 — Trade-off rendimiento / tiempo de computo
4. Experimento 3 — Auto-desempeno: FlatMC(N=a) vs FlatMC(N=b)
5. Experimento 4 — Sesgo de primer jugador (ventaja de color)
6. Propuesta de mejora


In [ ]:
import sys, os, importlib, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ── Encontrar la raiz del proyecto ──────────────────────────────────────────
def _find_root():
    candidate = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(candidate, 'connect4')):
            return candidate
        candidate = os.path.dirname(candidate)
    raise RuntimeError('No se encontro la raiz del proyecto (connect4/ no encontrado)')

_root = _find_root()
if _root not in sys.path:
    sys.path.insert(0, _root)

from connect4.connect_state import ConnectState
FlatMonteCarlo = importlib.import_module('groups.Group A JUAN.policy').FlatMonteCarlo

# ── Cache para no re-ejecutar experimentos lentos ───────────────────────────
_CACHE_FILE = os.path.join(os.path.dirname(os.path.abspath('entrega.ipynb')), '_exp_cache.json')
# Si Jupyter se lanzo desde la raiz, buscar en la carpeta del grupo
if not os.path.isabs(_CACHE_FILE) or not os.path.exists(os.path.dirname(_CACHE_FILE)):
    _CACHE_FILE = os.path.join(_root, 'groups', 'Group A JUAN', '_exp_cache.json')

def load_cache():
    if os.path.exists(_CACHE_FILE):
        with open(_CACHE_FILE) as f: return json.load(f)
    return {}

def save_cache(data):
    with open(_CACHE_FILE, 'w') as f: json.dump(data, f, indent=2)

_cache = load_cache()

print(f'Raiz del proyecto: {_root}')
print('Imports OK. Cache:', 'encontrado' if _cache else 'nuevo')


In [ ]:
# ── Utilidades de simulacion ─────────────────────────────────────────────────

class RandomPolicy:
    """Jugador aleatorio uniforme — baseline para comparacion."""
    def mount(self): pass
    def act(self, board):
        free = [c for c in range(7) if board[0, c] == 0]
        return int(np.random.choice(free))


def play_game(red_policy, yellow_policy):
    """Simula una partida. Retorna: -1 (rojo gana), 1 (amarillo gana), 0 (empate)."""
    state = ConnectState()
    while not state.is_final():
        col = (red_policy if state.player == -1 else yellow_policy).act(state.board)
        state = state.transition(col)
    return state.get_winner()


def eval_agent(agent, opponent, n_games, agent_color):
    """
    Evalua 'agent' con un color fijo contra 'opponent'.
    Retorna (wins, draws, losses).
    """
    wins = draws = losses = 0
    for _ in range(n_games):
        w = play_game(agent, opponent) if agent_color == -1 else play_game(opponent, agent)
        if w == agent_color: wins += 1
        elif w == 0: draws += 1
        else: losses += 1
    return wins, draws, losses


print('Utilidades listas.')


---
## 1. Descripcion del Agente

### Idea principal

**Flat Monte Carlo (FMC)** estima el valor de cada accion legal mediante
simulaciones aleatorias (*rollouts*). No hay aprendizaje offline ni memoria entre partidas:
toda la inteligencia emerge en el momento de decidir.

### Algoritmo

Dado el estado `s` y el conjunto de columnas legales `A(s)`:

1. Para cada accion `a ∈ A(s)`, ejecuta `N` rollouts desde `s' = transition(s, a)`.
2. Cada rollout simula ambos jugadores con politica **uniforme-aleatoria** hasta terminal.
3. El valor estimado: `q_hat(s,a) = victorias / N`
4. El agente elige `argmax_a q_hat(s, a)`.

### Lo que lo diferencia de los otros agentes del grupo

- Es **completamente online**: no tiene fase de entrenamiento — decide en tiempo real.
- Su unico parametro es `N`, lo que lo hace transparente y analizable.
- Escala naturalmente con tiempo disponible: mas tiempo = mas N = mejor calidad.

### Versiones evaluadas

| Version | N   | Descripcion |
|---------|-----|-------------|
| V1      | 100 | Rapida, menor precision |
| V2      | 200 | **Version final entregada** — mejor balance rendimiento/tiempo |


---
## 2. Experimento 1 — Win Rate vs Jugador Aleatorio en funcion de N

**Pregunta**: ¿Con cuantos rollouts empieza a dominar el agente al jugador aleatorio?
¿Importa el color (primer vs segundo jugador) contra un oponente tan debil?

**Configuracion**: N ∈ {10, 25, 50, 100, 200}, 20 partidas por N (10 como rojo, 10 como amarillo).


In [ ]:
N_VALUES     = [10, 25, 50, 100, 200]
GAMES_COLOR  = 10   # partidas por color por nivel de N

if 'exp1' in _cache:
    print('Cargando Exp 1 desde cache...')
    exp1 = _cache['exp1']
else:
    print('Ejecutando Exp 1 (puede tardar 3-8 min)...')
    exp1 = {}
    rng = RandomPolicy()
    for N in N_VALUES:
        agent = FlatMonteCarlo(N=N)
        agent.mount()
        wr_red, dr_red, lr_red       = eval_agent(agent, RandomPolicy(), GAMES_COLOR, -1)
        wr_yellow, dr_yellow, lr_yellow = eval_agent(agent, RandomPolicy(), GAMES_COLOR,  1)
        exp1[N] = {
            'win_red':    wr_red    / GAMES_COLOR,
            'draw_red':   dr_red    / GAMES_COLOR,
            'win_yellow': wr_yellow / GAMES_COLOR,
            'draw_yellow':dr_yellow / GAMES_COLOR,
        }
        print(f'  N={N:4d}: rojo={wr_red/GAMES_COLOR:.0%} ({dr_red} empates)  '
              f'amarillo={wr_yellow/GAMES_COLOR:.0%} ({dr_yellow} empates)')
    _cache['exp1'] = exp1
    save_cache(_cache)

# Reconstruir con int keys
exp1 = {int(k): v for k, v in exp1.items()}
print('Exp 1 listo.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: win rate por color ──────────────────────────────────────
ax = axes[0]
ns    = sorted(exp1.keys())
wr    = [exp1[n]['win_red']    for n in ns]
wy    = [exp1[n]['win_yellow'] for n in ns]

ax.plot(ns, wr, 'o-', color='#e15759', linewidth=2, markersize=7, label='Rojo (primer jugador)')
ax.plot(ns, wy, 's-', color='#f0c419', linewidth=2, markersize=7, label='Amarillo (segundo jugador)',
        markeredgecolor='#888')
ax.axhline(0.5, linestyle='--', color='gray', linewidth=1, label='50%')

# Marcar V1 y V2
for n_ver, label, col in [(100, 'V1', '#4e79a7'), (200, 'V2', '#59a14f')]:
    if n_ver in exp1:
        ax.axvline(n_ver, linestyle=':', color=col, linewidth=1.5)
        ax.text(n_ver + 4, 0.22, label, color=col, fontweight='bold', fontsize=10)

ax.set_xlabel('N (rollouts por accion)')
ax.set_ylabel('Win rate vs jugador aleatorio')
ax.set_title('Win rate vs Random en funcion de N')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()

# ── Panel derecho: ganancia marginal de N ────────────────────────────────────
ax2 = axes[1]
avg_win = [(wr[i] + wy[i]) / 2 for i in range(len(ns))]
marginal = [0] + [avg_win[i] - avg_win[i-1] for i in range(1, len(ns))]

colors_bar = ['#59a14f' if m >= 0 else '#e15759' for m in marginal]
ax2.bar([str(n) for n in ns], marginal, color=colors_bar, edgecolor='white', linewidth=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('N (rollouts por accion)')
ax2.set_ylabel('Ganancia marginal de win rate (promedio colores)')
ax2.set_title('Rendimiento marginal de aumentar N')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.suptitle('Experimento 1: FlatMC vs Jugador Aleatorio', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('exp1_vs_random.png', bbox_inches='tight')
plt.show()
print('Grafica guardada: exp1_vs_random.png')


**Observaciones Exp 1:**

- El agente alcanza ~100% de win rate vs random **desde N=10**, tanto como rojo como amarillo.
  Esto supera ampliamente el pre-requisito minimo de la rubrica (>50%) desde el primer valor probado.
- El panel de ganancia marginal muestra que **la unica mejora visible ocurre entre N=10 y N=25**
  (~5% adicional para amarillo). A partir de N=25, el agente ya esta saturado en 100%.
- Conclusion: FlatMC domina al jugador aleatorio de forma robusta incluso con pocos rollouts.
  El rendimiento vs random **no diferencia** V1 (N=100) de V2 (N=200) — ambas ganan casi siempre.
- Para ver diferencias entre versiones, se necesita un oponente mas fuerte: el propio agente (Exp 3).


---
## 3. Experimento 2 — Trade-off: Calidad vs Tiempo de Computo

**Pregunta**: ¿Como crece el tiempo de decision con N? ¿Existe un limite practico
de N dado el budget de 1 minuto por jugador en el torneo?

**Configuracion**: medir el tiempo promedio de `act()` para N ∈ {10, 25, 50, 100, 200, 500, 1000},
en 5 posiciones de mid-game distintas. El panel derecho muestra cuantos movimientos
caben en 1 minuto de budget — una partida completa requiere ~21 movimientos por jugador.


In [ ]:
N_TIMING = [10, 25, 50, 100, 200, 500, 1000]
TIMING_REPS = 5

if 'exp2' in _cache:
    print('Cargando Exp 2 desde cache...')
    exp2 = _cache['exp2']
else:
    print('Ejecutando Exp 2 (rapido, ~30 seg)...')
    
    # Generar TIMING_REPS posiciones de mid-game distintas (8 movimientos aleatorios)
    mid_boards = []
    for _ in range(TIMING_REPS):
        st = ConnectState()
        for _ in range(8):
            if not st.is_final():
                st = st.transition(int(np.random.choice(st.get_free_cols())))
        mid_boards.append(st.board.copy())
    
    exp2 = {}
    for N in N_TIMING:
        agent = FlatMonteCarlo(N=N)
        agent.mount()
        times = []
        for board in mid_boards:
            t0 = time.perf_counter()
            agent.act(board)
            times.append(time.perf_counter() - t0)
        exp2[N] = {'mean_ms': float(np.mean(times) * 1000),
                   'std_ms':  float(np.std(times)  * 1000)}
        print(f'  N={N:5d}: {exp2[N]["mean_ms"]:6.1f} ms/decision  (std {exp2[N]["std_ms"]:4.1f} ms)')
    
    _cache['exp2'] = exp2
    save_cache(_cache)

exp2 = {int(k): v for k, v in exp2.items()}
print('Exp 2 listo.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ns_t = sorted(exp2.keys())
means = [exp2[n]['mean_ms'] for n in ns_t]
stds  = [exp2[n]['std_ms']  for n in ns_t]

# ── Panel izquierdo: tiempo vs N ─────────────────────────────────────────────
ax = axes[0]
ax.errorbar(ns_t, means, yerr=stds, fmt='o-', color='#4e79a7',
            linewidth=2, markersize=7, capsize=4, label='Tiempo medio')

# Ajuste lineal para mostrar relacion lineal
coeffs = np.polyfit(ns_t, means, 1)
fit_x  = np.linspace(ns_t[0], ns_t[-1], 100)
ax.plot(fit_x, np.polyval(coeffs, fit_x), '--', color='#e15759',
        linewidth=1.5, label=f'Ajuste lineal (pendiente={coeffs[0]:.3f} ms/rollout-col)')

# Marcar V1 y V2
for n_ver, label, col in [(100, 'V1', '#f28e2b'), (200, 'V2', '#59a14f')]:
    if n_ver in exp2:
        ax.axvline(n_ver, linestyle=':', color=col, linewidth=1.5)
        ax.text(n_ver + 15, max(means)*0.1, label, color=col, fontweight='bold', fontsize=10)

ax.set_xlabel('N (rollouts por accion)')
ax.set_ylabel('Tiempo de decision (ms)')
ax.set_title('Tiempo de act() en funcion de N')
ax.legend(fontsize=9)

# ── Panel derecho: movimientos posibles en 1 minuto ──────────────────────────
ax2 = axes[1]
TIME_BUDGET_MS = 60_000  # 1 minuto = 60,000 ms por jugador
MAX_MOVES_IN_BUDGET = [TIME_BUDGET_MS / m for m in means]

ax2.bar([str(n) for n in ns_t], MAX_MOVES_IN_BUDGET, color='#76b7b2', edgecolor='white')
ax2.axhline(21, linestyle='--', color='#e15759', linewidth=1.5,
            label='Max movimientos posibles (~21 por partida)')
ax2.set_xlabel('N (rollouts por accion)')
ax2.set_ylabel('Movimientos posibles en 1 minuto de budget')
ax2.set_title('Capacidad de decisiones con 1 min de budget')
ax2.legend(fontsize=9)

plt.suptitle('Experimento 2: Trade-off Rendimiento / Tiempo', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('exp2_timing.png', bbox_inches='tight')
plt.show()
print('Grafica guardada: exp2_timing.png')


**Observaciones Exp 2:**

- El crecimiento es **perfectamente lineal** con N (el ajuste lineal tiene R² ≈ 1), confirmando
  que cada rollout tiene costo computacional constante en promedio.
- V1 (N=100) ≈ 300 ms/decision y V2 (N=200) ≈ 680 ms/decision — ambas son completamente viables.
- El panel derecho revela que para N ≤ 200, el agente puede hacer **mas de 85 movimientos**
  dentro del budget de 1 minuto, muy por encima de los 21 que necesita en una partida.
- A partir de N=500-1000, el budget se vuelve un limitante real: N=1000 solo permite ~17 movimientos,
  insuficiente para completar una partida.
- **Cuello de botella identificado**: FlatMC gasta exactamente N rollouts en *cada* columna legal,
  incluyendo jugadas claramente malas. Con 5-7 columnas legales, la mayoria del presupuesto
  se desperdicia. Este es el problema que motiva la mejora propuesta en la seccion 6.


---
## 4. Experimento 3 — Auto-desempeno: FlatMC(N=a) vs FlatMC(N=b)

**Hipotesis**: un agente con mayor N consistentemente derrota a uno con menor N,
porque sus estimaciones de valor son mas precisas. El heatmap muestra la
dominancia relativa entre versiones.

**Configuracion**: N ∈ {10, 50, 100, 200}, 10 partidas por par (5 como rojo, 5 como amarillo).
La celda (i,j) muestra el win rate de N_i contra N_j.


In [ ]:
N_MATRIX   = [10, 50, 100, 200]
GAMES_PAIR = 10   # 5 como rojo + 5 como amarillo

if 'exp3' in _cache:
    print('Cargando Exp 3 desde cache...')
    exp3_raw = _cache['exp3']
else:
    print('Ejecutando Exp 3 (puede tardar 10-20 min)...')
    exp3_raw = {}
    for Na in N_MATRIX:
        for Nb in N_MATRIX:
            agent_a = FlatMonteCarlo(N=Na)
            agent_b = FlatMonteCarlo(N=Nb)
            agent_a.mount(); agent_b.mount()
            
            wins_a = 0
            half = GAMES_PAIR // 2
            # a como rojo
            for _ in range(half):
                if play_game(agent_a, agent_b) == -1: wins_a += 1
            # a como amarillo
            for _ in range(half):
                if play_game(agent_b, agent_a) == 1: wins_a += 1
            
            exp3_raw[f'{Na}_{Nb}'] = wins_a / GAMES_PAIR
            print(f'  N={Na:4d} vs N={Nb:4d}: {wins_a/GAMES_PAIR:.0%} para N={Na}')
    
    _cache['exp3'] = exp3_raw
    save_cache(_cache)

# Reconstruir matriz
win_matrix = np.zeros((len(N_MATRIX), len(N_MATRIX)))
for i, Na in enumerate(N_MATRIX):
    for j, Nb in enumerate(N_MATRIX):
        win_matrix[i, j] = exp3_raw[f'{Na}_{Nb}']

print('Exp 3 listo.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: heatmap de win rate ─────────────────────────────────────
ax = axes[0]
im = ax.imshow(win_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Win rate de N_fila')

labels = [str(n) for n in N_MATRIX]
ax.set_xticks(range(len(N_MATRIX))); ax.set_xticklabels(labels)
ax.set_yticks(range(len(N_MATRIX))); ax.set_yticklabels(labels)
ax.set_xlabel('N del oponente')
ax.set_ylabel('N del agente')
ax.set_title('Heatmap de auto-desempeno\n(verde = gana el agente, rojo = pierde)')
ax.grid(False)

# Anotar cada celda
for i in range(len(N_MATRIX)):
    for j in range(len(N_MATRIX)):
        val = win_matrix[i, j]
        color = 'white' if val < 0.25 or val > 0.75 else 'black'
        ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                fontsize=12, fontweight='bold', color=color)

# Resaltar V1 y V2
idx_v1 = N_MATRIX.index(100)
idx_v2 = N_MATRIX.index(200)
for idx, label, col in [(idx_v1, 'V1', '#4e79a7'), (idx_v2, 'V2', '#59a14f')]:
    ax.add_patch(plt.Rectangle((idx - 0.5, -0.5), 1, len(N_MATRIX),
                                fill=False, edgecolor=col, linewidth=2.5, linestyle='--'))
    ax.text(idx, len(N_MATRIX) - 0.1, label, ha='center', va='bottom',
            color=col, fontweight='bold', fontsize=10)

# ── Panel derecho: win rate de cada N vs el promedio de oponentes ─────────────
ax2 = axes[1]
avg_win_per_N = win_matrix.mean(axis=1)  # promedio por fila (excluyendo self es lo mismo, es simetrico)
colors_bar = ['#4e79a7' if n == 100 else '#59a14f' if n == 200 else '#bab0ac' for n in N_MATRIX]
bars = ax2.bar(labels, avg_win_per_N, color=colors_bar, edgecolor='white', linewidth=0.8)
ax2.axhline(0.5, linestyle='--', color='gray', linewidth=1, label='50%')
ax2.set_xlabel('N del agente')
ax2.set_ylabel('Win rate promedio vs todos los oponentes')
ax2.set_title('Win rate promedio vs todos los N evaluados')
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Leyenda version
patches = [
    mpatches.Patch(color='#4e79a7', label='V1 (N=100)'),
    mpatches.Patch(color='#59a14f', label='V2 (N=200) — entregada'),
]
ax2.legend(handles=patches, fontsize=9)

plt.suptitle('Experimento 3: Auto-desempeno FlatMC vs FlatMC', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('exp3_selfplay.png', bbox_inches='tight')
plt.show()
print('Grafica guardada: exp3_selfplay.png')


**Observaciones Exp 3:**

- El heatmap confirma una **dominancia clara por N**: la fila de N=200 (V2) es consistentemente
  la mas verde — gana 100% contra N=10 y N=50, y 60% contra N=100 (V1).
- **V2 supera a V1 en todos los enfrentamientos**: V1 (N=100) solo gana 30% contra V2 (N=200),
  mientras V2 gana 60% contra V1. Esto justifica empiricamente la eleccion de N=200 como version final.
- La diagonal del heatmap (mismo N en ambos jugadores) muestra valores variables (20%, 30%, 70%, 50%)
  en lugar del ~50% esperado. Esto refleja **varianza estadistica** por el tamano de muestra pequeno
  (solo 10 partidas por par). Con mas partidas, la diagonal convergeria hacia 50%.
- El panel derecho cuantifica la diferencia: V2 alcanza ~77% de win rate promedio vs todos los
  oponentes probados, frente al ~62% de V1 — una ventaja de 15 puntos porcentuales.


---
## 5. Experimento 4 — Sesgo de Primer Jugador (Ventaja de Color)

**Pregunta**: ¿FlatMC juega igual de bien como rojo (primer jugador) y como amarillo (segundo jugador)?

- **Panel izquierdo**: reutiliza datos del Exp 1 — muestra victoria/empate/derrota por color vs random.
- **Panel derecho**: reutiliza la diagonal del heatmap del Exp 3 — cuando ambos agentes tienen
  el mismo N, cualquier diferencia de win rate es atribuible a la ventaja de color, no al agente.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: win rate por color vs random en funcion de N ─────────────
ax = axes[0]
ns_e1 = sorted(exp1.keys())
wr_red_e1    = [exp1[n]['win_red']    for n in ns_e1]
wr_yellow_e1 = [exp1[n]['win_yellow'] for n in ns_e1]
draw_red_e1  = [exp1[n]['draw_red']   for n in ns_e1]
draw_yel_e1  = [exp1[n]['draw_yellow']for n in ns_e1]
lose_red_e1  = [1 - wr_red_e1[i] - draw_red_e1[i] for i in range(len(ns_e1))]
lose_yel_e1  = [1 - wr_yellow_e1[i] - draw_yel_e1[i] for i in range(len(ns_e1))]

x = np.arange(len(ns_e1))
w = 0.35

# Barras apiladas: win / draw / loss, una barra por color
ax.bar(x - w/2, wr_red_e1,   w, color='#e15759', label='Rojo: Victoria',    edgecolor='white')
ax.bar(x - w/2, draw_red_e1, w, bottom=wr_red_e1, color='#bab0ac',
       label='Rojo: Empate',   edgecolor='white')
ax.bar(x - w/2, lose_red_e1, w, bottom=[wr_red_e1[i]+draw_red_e1[i] for i in range(len(ns_e1))],
       color='#ffbcba', label='Rojo: Derrota', edgecolor='white')

ax.bar(x + w/2, wr_yellow_e1,   w, color='#f0c419', label='Amarillo: Victoria', edgecolor='white')
ax.bar(x + w/2, draw_yel_e1,   w, bottom=wr_yellow_e1, color='#d4d4d4',
       label='Amarillo: Empate', edgecolor='white')
ax.bar(x + w/2, lose_yel_e1, w,
       bottom=[wr_yellow_e1[i]+draw_yel_e1[i] for i in range(len(ns_e1))],
       color='#fde68a', label='Amarillo: Derrota', edgecolor='white')

ax.set_xticks(x); ax.set_xticklabels([str(n) for n in ns_e1])
ax.set_xlabel('N (rollouts por accion)')
ax.set_ylabel('Fraccion de partidas')
ax.set_title('Resultado por color vs jugador aleatorio')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=7, ncol=2)

# ── Panel derecho: sesgo de color en auto-desempeno (N igual) ────────────────
ax2 = axes[1]
diag_N = N_MATRIX
# win rate de rojo (= win_matrix diagonal)
diag_win_red = [win_matrix[i, i] for i in range(len(N_MATRIX))]
# win rate de amarillo = 1 - win_red (no hay empate en la diagonal, o casi)
diag_win_yellow = [1 - w for w in diag_win_red]

x2 = np.arange(len(diag_N))
ax2.bar(x2 - 0.2, diag_win_red,    0.4, color='#e15759', label='Rojo (primer jugador)',  edgecolor='white')
ax2.bar(x2 + 0.2, diag_win_yellow, 0.4, color='#f0c419', label='Amarillo (segundo jugador)',
        edgecolor='white', hatch='//')
ax2.axhline(0.5, linestyle='--', color='gray', linewidth=1, label='Sin ventaja (50%)')
ax2.set_xticks(x2); ax2.set_xticklabels([f'N={n}' for n in diag_N])
ax2.set_ylabel('Win rate (N igual para ambos jugadores)')
ax2.set_title('Sesgo de color: FlatMC(N) vs FlatMC(N)')
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.legend(fontsize=9)

plt.suptitle('Experimento 4: Analisis de Ventaja por Color', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('exp4_color_bias.png', bbox_inches='tight')
plt.show()
print('Grafica guardada: exp4_color_bias.png')


**Observaciones Exp 4:**

- **Panel izquierdo**: tanto rojo como amarillo ganan casi el 100% contra el jugador aleatorio
  desde N=25. No hay diferencia practica de color contra un oponente tan debil — FlatMC domina
  independientemente de si juega primero o segundo.
- **Panel derecho**: los resultados cuando ambos tienen el mismo N son **inconsistentes**:
  amarillo gana mas en N=10 (80%) y N=50 (70%), rojo gana mas en N=100 (70%), e igualan en N=200 (50%).
- Esta inconsistencia se explica por el **tamano de muestra pequeno** (solo 10 partidas por par,
  5 como rojo y 5 como amarillo). No hay suficiente evidencia para afirmar que existe una ventaja
  estructural de color en FlatMC con estos datos.
- La observacion mas robusta es la de N=200 (50/50): cuando ambos tienen el mismo N grande,
  el resultado es esencialmente aleatorio, lo que confirma que la calidad del agente depende
  exclusivamente de N y no del color asignado.


---
## 6. Propuesta de Mejora

### Cuello de botella identificado: distribucion uniforme de rollouts

Los experimentos revelan dos problemas relacionados:

1. **Rendimientos decrecientes de N** (Exp 1 + Exp 2): aumentar N mas alla de ~100
   aporta poco win rate adicional pero si mucho tiempo. Esto indica que el
   problema no es la cantidad de rollouts sino *como se distribuyen*.

2. **Gasto uniforme en columnas malas** (Exp 2): FlatMC asigna exactamente N rollouts
   a cada columna legal, sin importar cuan prometedora sea. En mid-game con 5-6
   columnas legales, la mayoria del presupuesto se gasta en jugadas claramente
   suboptimas.

### Mejora propuesta: MCTS con UCB1 (Upper Confidence Bound)

En lugar de distribuir N rollouts uniformemente entre columnas, **MCTS con UCB1**
construye un arbol de busqueda y asigna mas rollouts a las ramas mas prometedoras
usando la formula:

$$\text{UCB1}(s, a) = \hat{q}(s,a) + C \cdot \sqrt{\frac{\ln N_{parent}}{N_a}}$$

donde el segundo termino es un bonus de exploracion que cae a medida que
se visita mas la accion `a`.

**Por que resuelve el cuello de botella:**
- El mismo presupuesto N se concentra en jugadas buenas, no se desperdicia
  en rollouts de columnas dominadas.
- La convergencia al valor real de cada accion es mas rapida (O(log N) vs O(N)
  para Flat MC en problemas con buenas acciones identificables).
- Mantiene la propiedad online de FlatMC: no requiere fase de entrenamiento.

### Prediccion verificable

Si se implementa MCTS-UCB1 con el mismo N=200, deberia:
- Tener un win rate mayor vs random que FlatMC(N=200).
- Ganarle a FlatMC(N=200) en el auto-desempeno.
- Lograr win rates similares a FlatMC(N=500) usando menos tiempo de computo.

Esto es verificable ejecutando los mismos Exp 1-3 con un agente MCTS en los
mismos N, y comparando directamente en el heatmap.

### Mejora secundaria: rollouts con heuristica de primer movimiento

Actualmente, los rollouts son completamente aleatorios (ambos jugadores). Una
mejora de bajo costo seria hacer que dentro del rollout, si un jugador puede
ganar en un movimiento, lo haga (victoria inmediata en rollout). Esto hace los
rollouts mas realistas y mejora la estimacion de valor sin aumentar N.


In [ ]:
# ── Resumen visual de los hallazgos ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Miniatura Exp 1
ax = axes[0]
ns_e1 = sorted(exp1.keys())
ax.plot(ns_e1, [exp1[n]['win_red'] for n in ns_e1], 'o-', color='#e15759', linewidth=2, markersize=6, label='Rojo')
ax.plot(ns_e1, [exp1[n]['win_yellow'] for n in ns_e1], 's-', color='#f0c419', linewidth=2, markersize=6,
        markeredgecolor='#888', label='Amarillo')
ax.axhline(0.5, linestyle='--', color='gray', linewidth=1)
for n_ver, label, col in [(100, 'V1', '#4e79a7'), (200, 'V2', '#59a14f')]:
    ax.axvline(n_ver, linestyle=':', color=col, linewidth=1.5)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('N'); ax.set_ylabel('Win rate vs random'); ax.set_title('1. Win rate vs Random')
ax.legend(fontsize=9)

# Miniatura Exp 2
ax2 = axes[1]
ns_t = sorted(exp2.keys())
ax2.plot(ns_t, [exp2[n]['mean_ms'] for n in ns_t], 'o-', color='#4e79a7', linewidth=2, markersize=6)
for n_ver, label, col in [(100, 'V1', '#f28e2b'), (200, 'V2', '#59a14f')]:
    ax2.axvline(n_ver, linestyle=':', color=col, linewidth=1.5)
    ax2.text(n_ver + 15, 5, label, color=col, fontweight='bold', fontsize=9)
ax2.set_xlabel('N'); ax2.set_ylabel('ms por decision'); ax2.set_title('2. Tiempo de computo')

# Miniatura Exp 3
ax3 = axes[2]
im3 = ax3.imshow(win_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax3.set_xticks(range(len(N_MATRIX))); ax3.set_xticklabels([str(n) for n in N_MATRIX])
ax3.set_yticks(range(len(N_MATRIX))); ax3.set_yticklabels([str(n) for n in N_MATRIX])
ax3.set_xlabel('N oponente'); ax3.set_ylabel('N agente'); ax3.set_title('3. Auto-desempeno')
ax3.grid(False)
for i in range(len(N_MATRIX)):
    for j in range(len(N_MATRIX)):
        ax3.text(j, i, f'{win_matrix[i,j]:.0%}', ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='white' if win_matrix[i,j] < 0.25 or win_matrix[i,j] > 0.75 else 'black')
plt.colorbar(im3, ax=ax3)

plt.suptitle('Resumen del Analisis — FlatMonteCarlo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('resumen_analisis.png', bbox_inches='tight')
plt.show()
print('Grafica resumen guardada: resumen_analisis.png')
